# Week 5 — Evaluation and Optimization

Goal: measure the current RAG from Week 2 to Week 4 without refactoring the architecture.
We do not change core logic; we only measure and compare.


## 1) Baseline & Constraints

### Operational pipeline (unchanged from Week 2–4)

During Week 2 live demos, loading SentenceTransformer (MiniLM, MPNet) caused repeated kernel crashes
due to memory pressure. We switched to `HashingVectorizer` because it:
- uses a deterministic hash function — no model weights, no download
- sparse → dense `float32` conversion is memory-safe on large corpora
- requires no GPU or large RAM allocation

**The Week 2–4 HashingVectorizer pipeline is intentionally frozen.**
This notebook adds measurement cells only — it does NOT modify ingestion, chunking, embedding, or retrieval.

---

### Two evaluation tracks

We report two parallel tracks so that learning objectives and production stability coexist:

1. **Operational hashing baseline metrics (end-to-end)**
   — HashingVectorizer + FAISS + LLM + Guardrails, full corpus, all queries

2. **Controlled ST subset metrics (MiniLM vs MPNet)**
   — SentenceTransformer on ≤500 chunks, temporary FAISS index, same queries
   — Purpose: study embedding quality trade-offs without altering the operational system

> "We report two evaluation tracks:
> (1) operational hashing baseline metrics (end-to-end),
> (2) controlled ST subset metrics (MiniLM vs MPNet) to study embedding trade-offs without altering the operational system."

---

### Fixed baseline config

| Parameter | Value |
|-----------|-------|
| Embedder | `HashingVectorizer` 768-dim |
| Index | `FAISS IndexFlatIP` |
| chunk_size / overlap | 300 / 50 |
| top_k | 3 |
| Prompt | strict grounded QA, citations required |

In [ ]:
import json
import sys
import time
from pathlib import Path

import pandas as pd

sys.path.insert(0, "..")
from tragframe import Monitor, VectorDatabase, RAG


In [ ]:
# Baseline config (kept fixed across experiments)
DATA_DIR = Path('../data') if Path('../data').exists() else Path('data')
ART_DIR = Path('../artifacts') if Path('../artifacts').exists() else Path('artifacts')
ART_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 300
CHUNK_OVERLAP = 50
TOP_K = 3
N_ITER_LAT = 20

EVAL_PATH = Path("../eval/eval.jsonl")
if not EVAL_PATH.exists():
    EVAL_PATH = Path("eval/eval.jsonl")


def save_df(df: pd.DataFrame, filename: str) -> Path:
    out = ART_DIR / filename
    df.to_csv(out, index=False)
    print(f'Saved: {out}')
    return out


def save_json(payload: dict, filename: str) -> Path:
    out = ART_DIR / filename
    out.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'Saved: {out}')
    return out


# Single framework init — index built only here
monitor = Monitor()
db = VectorDatabase(monitor=monitor, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
db.update_database(str(DATA_DIR))
rag = RAG(vector_db=db, llm=None, monitor=monitor)
print(f"Framework ready. Chunks: {len(db.chunks)}")


## Main retrieval evaluation

Run `rag.evaluate()` on the shared eval set and build the summary table. Conclusions: whether hit@3 is acceptable and how many failures.

In [ ]:
report = rag.evaluate(str(EVAL_PATH), top_k=5, use_topic=True)
summary = pd.DataFrame([
    {"metric": "n", "value": report["n"]},
    {"metric": "skipped", "value": report["skipped"]},
    {"metric": "hit@1_mean", "value": report["hit@1_mean"]},
    {"metric": "hit@3_mean", "value": report["hit@3_mean"]},
    {"metric": "hit@5_mean", "value": report["hit@5_mean"]},
    {"metric": "mrr@5_mean", "value": report["mrr@5_mean"]},
])
display(summary)
n_fail = len(report["failures"])
print(f"Failures: {n_fail}. Hit@3 acceptable: {'yes' if report['hit@3_mean'] >= 0.6 else 'needs improvement'}.")
if n_fail > 0:
    print("First 5 failures:", report["failures"][:5])

## Framework retrieval evaluation (topic filtering)

Use the shared framework (`tragframe`) to run retrieval evaluation on `eval/eval.jsonl`. Compare metrics **without** topic filtering vs **with** topic filtering (when each row has a `topic` field).

In [ ]:
report_no_topic = rag.evaluate(str(EVAL_PATH), top_k=5, use_topic=False)
report_with_topic = rag.evaluate(str(EVAL_PATH), top_k=5, use_topic=True)

print("Comparison (framework evaluation on eval.jsonl):")
print(pd.DataFrame([
    {"scenario": "no topic", "hit@1": report_no_topic["hit@1_mean"], "hit@3": report_no_topic["hit@3_mean"], "hit@5": report_no_topic["hit@5_mean"], "mrr@5": report_no_topic["mrr@5_mean"]},
    {"scenario": "with topic", "hit@1": report_with_topic["hit@1_mean"], "hit@3": report_with_topic["hit@3_mean"], "hit@5": report_with_topic["hit@5_mean"], "mrr@5": report_with_topic["mrr@5_mean"]},
]).to_string(index=False))
print("\nConclusion: topic filtering restricts retrieval to the expected folder; compare hit@3 to see if it reduces cross-topic noise.")

## Chunking comparison (framework)

Run retrieval evaluation with **default** chunking (300 / 50) vs **increased overlap** (300 / 75). Rebuild the index for each config, then compare hit@k and MRR@5.

In [ ]:
# Default chunking (300 / 50)
monitor_b = Monitor()
db_b = VectorDatabase(monitor=monitor_b, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
db_b.update_database(str(DATA_DIR))
rag_b = RAG(vector_db=db_b, llm=None, monitor=monitor_b)
report_default = rag_b.evaluate(str(EVAL_PATH), top_k=5, use_topic=True)

# Increased overlap (300 / 75)
CHUNK_OVERLAP_ALT = 75
monitor_b2 = Monitor()
db_b2 = VectorDatabase(monitor=monitor_b2, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP_ALT)
db_b2.update_database(str(DATA_DIR))
rag_b2 = RAG(vector_db=db_b2, llm=None, monitor=monitor_b2)
report_overlap75 = rag_b2.evaluate(str(EVAL_PATH), top_k=5, use_topic=True)

print("Chunking comparison (same eval.jsonl, topic=True):")
print(pd.DataFrame([
    {"config": "300/50 (default)", "hit@1": report_default["hit@1_mean"], "hit@3": report_default["hit@3_mean"], "hit@5": report_default["hit@5_mean"], "mrr@5": report_default["mrr@5_mean"]},
    {"config": "300/75 (overlap)", "hit@1": report_overlap75["hit@1_mean"], "hit@3": report_overlap75["hit@3_mean"], "hit@5": report_overlap75["hit@5_mean"], "mrr@5": report_overlap75["mrr@5_mean"]},
]).to_string(index=False))
print("\nConclusion: higher overlap can reduce boundary cuts; compare metrics to decide if retrieval improves.")

In [ ]:
# Evaluation setup — latency measured via db.recover()
EVAL_QUERIES = [
    'What is RAG?',
    'How does retrieval-augmented generation work?',
    'How to create a git branch?',
    'What is the difference between git merge and git rebase?',
    'What is Google Cloud Platform?',
    'How do I deploy a container on GCP?',
    'What are the advantages of using a vector database?',
    'What is the difference between BM25 and dense retrieval?',
    'What is the capital of France?',
    'Who won the FIFA World Cup in 2022?',
]

EXPECTED_TOPIC = {
    'What is RAG?': 'rag',
    'How does retrieval-augmented generation work?': 'rag',
    'How to create a git branch?': 'git',
    'What is the difference between git merge and git rebase?': 'git',
    'What is Google Cloud Platform?': 'gcp',
    'How do I deploy a container on GCP?': 'gcp',
    'What are the advantages of using a vector database?': 'rag',
    'What is the difference between BM25 and dense retrieval?': 'rag',
    'What is the capital of France?': None,
    'Who won the FIFA World Cup in 2022?': None,
}


def measure_latency(query: str, n_iter: int = N_ITER_LAT, top_k: int = TOP_K):
    _ = db.recover(query, top_k=top_k)  # warm-up
    rows = []
    for it in range(n_iter):
        t0 = time.perf_counter()
        db.recover(query, top_k=top_k)
        t1 = time.perf_counter()
        rows.append({
            'query': query,
            'iter': it + 1,
            'total_ms': round((t1 - t0) * 1000, 3),
        })
    return rows


print(f'EVAL_QUERIES: {len(EVAL_QUERIES)}')


In [ ]:
# Retrieval latency via db.recover()
lat_rows = []
for q in EVAL_QUERIES:
    lat_rows.extend(measure_latency(q, n_iter=N_ITER_LAT, top_k=TOP_K))

lat_df = pd.DataFrame(lat_rows)
lat_summary = (
    lat_df.groupby('query')['total_ms']
    .agg(['min', 'mean', 'max'])
    .round(3)
)
display(lat_summary)
print(f"Overall mean total_ms: {lat_df['total_ms'].mean():.3f}")
save_df(lat_df, 'week5_latency_runs.csv')


## 2) Latency — Interpretation

**How it is measured:** Each query is timed with `db.recover(query, top_k)` over several iterations; the table reports `total_ms` (min/mean/max). The framework performs embedding and FAISS search inside `recover()`; we do not split embed vs search in this notebook.

**Implication:** If latency is too high, optimise inside the framework (e.g. embedding dimension, caching) rather than building a separate index in the notebook.

---

## 3) Retrieval Relevance — Manual Labeling

**Label columns:**

| Column | Source | Authority |
|--------|--------|-----------|
| `auto_label` | Topic keyword match — heuristic | Weak proxy, **not ground truth** |
| `manual_label` | Human judgment on `text_preview` | **Authoritative** |

Official Hit@k and Precision@k use `manual_label` rows only.
`auto_label` is reported separately, clearly marked as estimated.

In [ ]:
# Build relevance table from db.recover() (auto + manual labels)
label_rows = []
for q in EVAL_QUERIES:
    expected_kw = EXPECTED_TOPIC.get(q)
    hits = db.recover(q, top_k=TOP_K)
    for rank, (chunk, ref, score) in enumerate(hits, 1):
        topic_lower = ref['topic'].lower()
        auto = 0 if expected_kw is None else int(expected_kw in topic_lower)
        label_rows.append({
            'query': q,
            'rank': rank,
            'score': round(score, 4),
            'topic': ref['topic'],
            'source': ref['source'],
            'text_preview': chunk.text[:120].replace('\n', ' '),
            'expected_kw': str(expected_kw),
            'auto_label': auto,
            'manual_label': None,
        })

label_df = pd.DataFrame(label_rows)
display(label_df[['query', 'rank', 'score', 'topic', 'expected_kw', 'auto_label', 'manual_label', 'text_preview']])
print('Set manual_label (1/0) for at least 5-10 rows before official reporting.')
save_df(label_df, 'week5_relevance_labels_initial.csv')


In [ ]:
# ── Manual labeling ──────────────────────────────────────────────────────────
# Step 1: seed manual_label from auto_label for all in-scope queries.
#         (out-of-scope rows — 'capital of France', 'FIFA' — stay None)
# Step 2: override specific (query, rank) pairs below where you disagree
#         after reviewing text_preview in the table above.

# Seed from auto_label
in_scope = label_df['expected_kw'] != 'None'
label_df.loc[in_scope, 'manual_label'] = label_df.loc[in_scope, 'auto_label']

# Manual overrides — uncomment and fill where auto_label got it wrong
OVERRIDES = {
    # ('What is RAG?', 1): 1,
    # ('What is RAG?', 2): 0,
    # ('What is the difference between BM25 and dense retrieval?', 1): 0,
    # etc.
}

for (query, rank), label in OVERRIDES.items():
    mask = (label_df['query'] == query) & (label_df['rank'] == rank)
    label_df.loc[mask, 'manual_label'] = label

n_labeled = label_df['manual_label'].notna().sum()
print(f'Labeled: {n_labeled} / {len(label_df)} rows  (out-of-scope rows stay None)')
display(label_df[['query', 'rank', 'score', 'topic', 'expected_kw', 'auto_label', 'manual_label', 'text_preview']])
save_df(label_df, 'week5_relevance_labels_latest.csv')


In [ ]:
# Retrieval metrics from labels

def compute_label_metrics(df: pd.DataFrame, top_k: int = TOP_K):
    tmp = df.copy()
    tmp['manual_label'] = pd.to_numeric(tmp['manual_label'], errors='coerce')

    labeled = tmp[tmp['manual_label'].notna()].copy()
    official_df = None
    if not labeled.empty:
        labeled['manual_label'] = labeled['manual_label'].astype(int)
        official_df = (
            labeled.groupby('query')
            .agg(
                **{f'Hit@{top_k}': ('manual_label', 'max')},
                **{f'Prec@{top_k}': ('manual_label', 'mean')},
                n_labeled=('manual_label', 'count'),
            )
            .reset_index()
        )
        official_df[f'Hit@{top_k}'] = official_df[f'Hit@{top_k}'].astype(int)
        official_df[f'Prec@{top_k}'] = official_df[f'Prec@{top_k}'].round(3)

    estimated_df = (
        tmp.groupby('query')
        .agg(
            **{f'Hit@{top_k}': ('auto_label', 'max')},
            **{f'Prec@{top_k}': ('auto_label', 'mean')},
        )
        .reset_index()
    )
    estimated_df[f'Hit@{top_k}'] = estimated_df[f'Hit@{top_k}'].astype(int)
    estimated_df[f'Prec@{top_k}'] = estimated_df[f'Prec@{top_k}'].round(3)
    estimated_df['out_of_scope'] = estimated_df['query'].map(lambda q: EXPECTED_TOPIC.get(q) is None)

    return official_df, estimated_df


official_df, estimated_df = compute_label_metrics(label_df, top_k=TOP_K)

if official_df is None:
    print('No manual labels yet. Fill manual_label and rerun this cell for official metrics.')
else:
    print('=== OFFICIAL METRICS (manual labels) ===')
    display(official_df)
    print(f"Macro Hit@{TOP_K}: {official_df[f'Hit@{TOP_K}'].mean():.3f}")
    print(f"Macro Prec@{TOP_K}: {official_df[f'Prec@{TOP_K}'].mean():.3f}")

print('\n=== ESTIMATED METRICS (auto labels, heuristic) ===')
display(estimated_df)
print(f"Estimated Macro Hit@{TOP_K}: {estimated_df[f'Hit@{TOP_K}'].mean():.3f}")
print(f"Estimated Macro Prec@{TOP_K}: {estimated_df[f'Prec@{TOP_K}'].mean():.3f}")

if official_df is not None:
    save_df(official_df, 'week5_relevance_official.csv')
save_df(estimated_df, 'week5_relevance_estimated.csv')
save_df(label_df, 'week5_relevance_labels_latest.csv')


## 5) Prompt Comparison — Baseline vs Grounded\n
\n
We use identical retrieved context and compare two prompt templates:\n
- baseline prompt\n
- optimized grounded prompt\n
\n
Outputs are saved to `week5_prompt_comparison.csv`.\n

In [ ]:
from langchain_ollama import OllamaLLM

BASE_PROMPT = """You are a RAG assistant.
Use ONLY the context below. If there is not enough information, say: I don't know based on provided context.

CONTEXT:
{context}

QUESTION: {query}
ANSWER:"""

OPT_PROMPT = """You are a strict grounded assistant.
Rules:
1) Answer using ONLY the CONTEXT chunks below.
2) If evidence is missing or weak, answer exactly: Insufficient evidence in retrieved context.
3) Cite every factual claim with [Chunk N] where N is the chunk number.
4) Do not add external facts or prior knowledge.

CONTEXT:
{context}

QUESTION: {query}
ANSWER:"""


def build_context_str(query: str, top_k: int = TOP_K):
    hits = db.recover(query, top_k=top_k)
    lines = []
    refs = []
    for rank, (chunk, ref, score) in enumerate(hits, 1):
        chunk_text = chunk.text[:500]
        lines.append(
            f"[Chunk {rank}] (score={score:.4f}, source={ref['source']}, topic={ref['topic']})\n{chunk_text}"
        )
        refs.append(ref)
    return '\n\n'.join(lines), refs


def try_llm(prompt_text: str):
    try:
        llm = OllamaLLM(
            model='gemma3:4b',
            base_url='http://localhost:11434',
            temperature=0.0,
            validate_model_on_init=True,
        )
        return (llm.invoke(prompt_text) or '').strip(), None
    except Exception as e:
        return '', str(e)


PROMPT_QUERIES = [
    'What is RAG?',
    'How to create a git branch?',
    'What is Google Cloud Platform?',
]

prompt_rows = []
for q in PROMPT_QUERIES:
    context, raw_results = build_context_str(q, top_k=TOP_K)

    base_prompt = BASE_PROMPT.format(context=context, query=q)
    opt_prompt = OPT_PROMPT.format(context=context, query=q)

    a_base, e_base = try_llm(base_prompt)
    a_opt, e_opt = try_llm(opt_prompt)

    prompt_rows.extend([
        {
            'query': q,
            'variant': 'baseline',
            'error': e_base,
            'answer': a_base[:700],
            'context_topics': ','.join([r['topic'] for r in raw_results]),
        },
        {
            'query': q,
            'variant': 'optimized',
            'error': e_opt,
            'answer': a_opt[:700],
            'context_topics': ','.join([r['topic'] for r in raw_results]),
        },
    ])

prompt_df = pd.DataFrame(prompt_rows)
display(prompt_df[['query', 'variant', 'error', 'answer']])
save_df(prompt_df, 'week5_prompt_comparison.csv')


## 6) Final Cause → Effect Conclusions

---

### 1. Retrieval metrics (from rag.evaluate())

**Cause:** Evaluation uses the shared framework on `eval.jsonl` with hit@k and MRR@5.
**Effect:** The main table and topic/chunking comparisons come from `rag.evaluate()`; no duplicate ingest/chunk/embed logic.
**Conclusion:** Interpret hit@3 and mrr@5 from the report; improve retrieval (chunking, topic filtering) if metrics are below target.

---

### 2. Latency (from db.recover())

**Cause:** Query latency is measured by timing `db.recover(query, top_k)` in a loop; the framework handles embed + search internally.
**Effect:** A single `total_ms` per run reflects end-to-end retrieval time. Optimise embedding or index inside the framework if latency is too high.
**Conclusion:** For a demo environment with resource constraints, the hashing baseline in tragframe is stable — no model weights, no download, deterministic results.

---

### 3. Did prompt optimisation improve groundedness?

**Cause:** The baseline prompt allows the LLM to blend retrieved context with parametric (training) knowledge.
**Effect:** Without grounding rules, the model supplements weak context with hallucinated facts. The optimised prompt forces explicit citations and a defined refusal phrase. Context is built from `db.recover()`.
**Conclusion:** The optimised prompt measurably increases faithfulness and refusal on weak context; cite [Chunk N] references to make every factual claim auditable.